# Imports

In [1]:
from pathlib import Path
print(Path.cwd())

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent)) # problem with dependency resolution (e.g. custom_builder) without this

/Users/mac/Documents/dev/ID2221/dic/Week 2


In [2]:
# Use delta features if needed (DeltaTable, etc.)
from delta import *
from custom_builder import builder
from log import *

# use the existing preconfigured builder to create the Spark session.
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f'current database: {spark.catalog.currentDatabase()}')
print(f'spark tables: {spark.catalog.listTables()}')

from pyspark.sql import functions as F

import json

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/16 11:09:05 WARN Utils: Your hostname, MacBook-Pro-som-tillhor-MAC.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.247 instead (on interface en0)
26/09/16 11:09:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/mac/Documents/dev/ID2221/dic/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/mac/.ivy2.5.2/cache
The jars for the packages stored in: /Users/mac/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-251a3da8-2324-44da-ae90-d0da7ed82f49;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-ap

current database: default
spark tables: [Table(name='air_quality', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='integrated_taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_zone_lookup', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='weather', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False)]


# Uncache Tables

In [3]:
spark.catalog.uncacheTable("default.air_quality")
spark.catalog.uncacheTable("default.taxi_trips")
spark.catalog.uncacheTable("default.taxi_zone_lookup")
spark.catalog.uncacheTable("default.weather")

## Run queries on uncached tables

### Task 2.1

In [38]:
result = spark.sql("""
    SELECT
        zone AS pu_zone,
        month(pu_datetime) AS month, 
        COUNT(*) AS row_count
    FROM default.taxi_trips
    LEFT JOIN default.taxi_zone_lookup
    ON taxi_trips.pu_location_id = taxi_zone_lookup.location_id
    GROUP BY pu_zone, month
""")
result.summary().show()
result.show()

+-------+--------------------+------------------+------------------+
|summary|             pu_zone|             month|         row_count|
+-------+--------------------+------------------+------------------+
|  count|                 270|               271|               271|
|   mean|                NULL|1.4575645756457565|10939.365313653136|
| stddev|                NULL|2.1749942465823655|27353.533786659147|
|    min|Allerton/Pelham G...|                 1|                 1|
|    25%|                NULL|                 1|                78|
|    50%|                NULL|                 1|               245|
|    75%|                NULL|                 1|              1393|
|    max|      Yorkville West|                12|            145240|
+-------+--------------------+------------------+------------------+

+--------------------+-----+---------+
|             pu_zone|month|row_count|
+--------------------+-----+---------+
|  Van Cortlandt Park|    1|       18|
|              

### Task 2.2

In [39]:
result = spark.sql(
"""
SELECT
    CASE
        WHEN prcp > 0 THEN 'greater_than_0'
        ELSE 'zero_or_null'
  	END AS column_group,
    COUNT(*) AS cnt,
    AVG(trip_distance) AS avg_trip_distance
FROM default.taxi_trips AS t
LEFT JOIN default.weather AS w
ON date_trunc('hour', t.pu_datetime) = w.datetime
GROUP BY
    CASE
        WHEN prcp > 0 THEN 'greater_than_0'
            ELSE 'zero_or_null'
    END;
"""
)
result.show()


+--------------+-------+------------------+
|  column_group|    cnt| avg_trip_distance|
+--------------+-------+------------------+
|greater_than_0| 424773| 3.470743355446981|
|  zero_or_null|2539795|3.6824345463125323|
+--------------+-------+------------------+



### Task 2.3

In [40]:
res = spark.sql("""
SELECT measurement, COUNT(county) AS trips 
FROM (taxi_trips AS t
LEFT JOIN taxi_zone_lookup AS tzl
ON t.pu_location_id = tzl.location_id) t
LEFT JOIN (
    SELECT *
    FROM (
        SELECT
            measurement,
            county AS aq_county,
            date_trunc("hour", datetime) as hr_datetime,
            ROW_NUMBER() OVER (
                PARTITION BY date_trunc("hour", datetime)
                ORDER BY datetime DESC
            ) AS rn
        FROM air_quality r
    )
    WHERE rn = 1
) aq
ON date_trunc('hour', pu_datetime) = hr_datetime AND county = aq_county
GROUP BY measurement
ORDER BY measurement, trips
""")
res.show()

+-----------+-------+
|measurement|  trips|
+-----------+-------+
|       NULL|2945398|
|        1.3|     20|
|        1.6|     13|
|        1.7|     31|
|        1.8|     41|
|        1.9|     19|
|        2.0|     24|
|        2.1|    135|
|        2.2|     39|
|        2.3|     36|
|        2.4|     51|
|        2.5|    137|
|        2.6|     93|
|        2.7|     84|
|        2.8|     27|
|        2.9|    141|
|        3.0|     62|
|        3.1|     89|
|        3.2|    115|
|        3.3|     71|
+-----------+-------+
only showing top 20 rows


### Task 2.5

In [41]:
res = spark.sql("""
SELECT date_format(pu_datetime, 'EEE') AS day, hour(pu_datetime) AS hour, COUNT(*) AS trips
    FROM taxi_trips
    GROUP BY hour, day
    ORDER BY day, trips DESC
;
""")
res.show()

+---+----+-----+
|day|hour|trips|
+---+----+-----+
|Fri|  18|29050|
|Fri|  17|28034|
|Fri|  19|26656|
|Fri|  16|25645|
|Fri|  15|25578|
|Fri|  14|24746|
|Fri|  22|24015|
|Fri|  13|22190|
|Fri|  23|22100|
|Fri|  21|21976|
|Fri|  20|21168|
|Fri|  12|21024|
|Fri|  11|19852|
|Fri|  10|19559|
|Fri|   9|18192|
|Fri|   8|17323|
|Fri|   7|13043|
|Fri|   0| 8804|
|Fri|   6| 6283|
|Fri|   1| 4805|
+---+----+-----+
only showing top 20 rows


### Task 2.6

In [42]:
res = spark.sql("""
SELECT date_format(pu_datetime, 'MMM') AS month, COUNT(*) AS trips
    FROM taxi_trips
    GROUP BY month
    ORDER BY month
;
""")
res.show()

+-----+-------+
|month|  trips|
+-----+-------+
|  Dec|     12|
|  Feb|      3|
|  Jan|2964553|
+-----+-------+



# Cache Tables

In [43]:
spark.catalog.cacheTable("default.air_quality")
spark.catalog.cacheTable("default.taxi_trips")
spark.catalog.cacheTable("default.taxi_zone_lookup")
spark.catalog.cacheTable("default.weather")

## Run queries on cached tables

### Task 2.1

In [67]:
result = spark.sql("""
    SELECT
        zone AS pu_zone,
        month(pu_datetime) AS month, 
        COUNT(*) AS row_count
    FROM default.taxi_trips
    LEFT JOIN default.taxi_zone_lookup
    ON taxi_trips.pu_location_id = taxi_zone_lookup.location_id
    GROUP BY pu_zone, month
""")
result.summary().show()
result.show()

+-------+--------------------+------------------+------------------+
|summary|             pu_zone|             month|         row_count|
+-------+--------------------+------------------+------------------+
|  count|                 270|               271|               271|
|   mean|                NULL|1.4575645756457565|10939.365313653136|
| stddev|                NULL|2.1749942465823655|27353.533786659147|
|    min|Allerton/Pelham G...|                 1|                 1|
|    25%|                NULL|                 1|                78|
|    50%|                NULL|                 1|               245|
|    75%|                NULL|                 1|              1393|
|    max|      Yorkville West|                12|            145240|
+-------+--------------------+------------------+------------------+

+--------------------+-----+---------+
|             pu_zone|month|row_count|
+--------------------+-----+---------+
|  Van Cortlandt Park|    1|       18|
|              

### Task 2.2

In [68]:
result = spark.sql(
"""
SELECT
    CASE
        WHEN prcp > 0 THEN 'greater_than_0'
        ELSE 'zero_or_null'
  	END AS column_group,
    COUNT(*) AS cnt,
    AVG(trip_distance) AS avg_trip_distance
FROM default.taxi_trips AS t
LEFT JOIN default.weather AS w
ON date_trunc('hour', t.pu_datetime) = w.datetime
GROUP BY
    CASE
        WHEN prcp > 0 THEN 'greater_than_0'
            ELSE 'zero_or_null'
    END;
"""
)
result.show()


+--------------+-------+------------------+
|  column_group|    cnt| avg_trip_distance|
+--------------+-------+------------------+
|greater_than_0| 424773| 3.470743355446981|
|  zero_or_null|2539795|3.6824345463125323|
+--------------+-------+------------------+



### Task 2.3

In [69]:
res = spark.sql("""
SELECT measurement, COUNT(county) AS trips 
FROM (taxi_trips AS t
LEFT JOIN taxi_zone_lookup AS tzl
ON t.pu_location_id = tzl.location_id) t
LEFT JOIN (
    SELECT *
    FROM (
        SELECT
            measurement,
            county AS aq_county,
            date_trunc("hour", datetime) as hr_datetime,
            ROW_NUMBER() OVER (
                PARTITION BY date_trunc("hour", datetime)
                ORDER BY datetime DESC
            ) AS rn
        FROM air_quality r
    )
    WHERE rn = 1
) aq
ON date_trunc('hour', pu_datetime) = hr_datetime AND county = aq_county
GROUP BY measurement
ORDER BY measurement, trips
""")
res.show()

+-----------+-------+
|measurement|  trips|
+-----------+-------+
|       NULL|2945398|
|        1.3|     20|
|        1.6|     13|
|        1.7|     31|
|        1.8|     41|
|        1.9|     19|
|        2.0|     24|
|        2.1|    135|
|        2.2|     39|
|        2.3|     36|
|        2.4|     51|
|        2.5|    137|
|        2.6|     93|
|        2.7|     84|
|        2.8|     27|
|        2.9|    141|
|        3.0|     62|
|        3.1|     89|
|        3.2|    115|
|        3.3|     71|
+-----------+-------+
only showing top 20 rows


### Task 2.5

In [70]:
res = spark.sql("""
SELECT date_format(pu_datetime, 'EEE') AS day, hour(pu_datetime) AS hour, COUNT(*) AS trips
    FROM taxi_trips
    GROUP BY hour, day
    ORDER BY day, trips DESC
;
""")
res.show()

+---+----+-----+
|day|hour|trips|
+---+----+-----+
|Fri|  18|29050|
|Fri|  17|28034|
|Fri|  19|26656|
|Fri|  16|25645|
|Fri|  15|25578|
|Fri|  14|24746|
|Fri|  22|24015|
|Fri|  13|22190|
|Fri|  23|22100|
|Fri|  21|21976|
|Fri|  20|21168|
|Fri|  12|21024|
|Fri|  11|19852|
|Fri|  10|19559|
|Fri|   9|18192|
|Fri|   8|17323|
|Fri|   7|13043|
|Fri|   0| 8804|
|Fri|   6| 6283|
|Fri|   1| 4805|
+---+----+-----+
only showing top 20 rows


### Task 2.6

In [71]:
res = spark.sql("""
SELECT date_format(pu_datetime, 'MMM') AS month, COUNT(*) AS trips
    FROM taxi_trips
    GROUP BY month
    ORDER BY month
;
""")
res.show()

+-----+-------+
|month|  trips|
+-----+-------+
|  Dec|     12|
|  Feb|      3|
|  Jan|2964553|
+-----+-------+

